---
## 12a-c. Camera-ready additions (reviewer comments)
Run after Section 12 (needs m1-m5, X_sp, y_s). Send me the printed outputs.

In [ ]:
# 12a. SMOTE vs SMOTETomek - are the two really identical? (Reviewer 2)
import numpy as np
Xs_, ys_ = SMOTE(sampling_strategy=0.05, random_state=42).fit_resample(X_train, y_train)
smt = SMOTETomek(smote=SMOTE(sampling_strategy=0.05, random_state=42), random_state=42)
Xt_, yt_ = smt.fit_resample(X_train, y_train)
print(f'original train  : {len(y_train)}  (seizure {int(y_train.sum())})')
print(f'after SMOTE     : {len(ys_)}  (seizure {int(ys_.sum())})')
print(f'after SMOTETomek: {len(yt_)}  (seizure {int(yt_.sum())})')
print('Tomek links removed:', len(ys_) - len(yt_))
print('training sets identical:', len(ys_)==len(yt_) and np.array_equal(Xs_, Xt_))
print('CHB-MIT predictions identical (m2 vs m4):', np.array_equal(m2.predict(X_test), m4.predict(X_test)))
print('CHB-MIT probabilities identical        :', np.allclose(m2.predict_proba(X_test)[:,1], m4.predict_proba(X_test)[:,1]))
print('Siena predictions identical            :', np.array_equal(m2.predict(X_sp), m4.predict(X_sp)))
# REAL Siena SMOTETomek row (replaces the estimated one in TABLE VI):
yp = m4.predict(X_sp); pp = m4.predict_proba(X_sp)[:,1]
print(pd.DataFrame([ev('SMOTETomek+RF', y_s, yp, pp)]).to_string(index=False))


In [ ]:
# 12b. Five seeds for ALL techniques + paired significance tests (Reviewers 2 and 3)
from scipy.stats import wilcoxon, ttest_rel
SEEDS = [42,123,456,789,999]
def build(name, s):
    if name=='Baseline RF':  return RF(s).fit(X_train,y_train)
    if name=='SMOTE+RF':
        X_,y_ = SMOTE(sampling_strategy=0.05,random_state=s).fit_resample(X_train,y_train); return RF(s).fit(X_,y_)
    if name=='BalancedRF':   return BalancedRandomForestClassifier(n_estimators=100,n_jobs=-1,random_state=s).fit(X_train,y_train)
    if name=='SMOTETomek+RF':
        X_,y_ = SMOTETomek(smote=SMOTE(sampling_strategy=0.05,random_state=s),random_state=s).fit_resample(X_train,y_train); return RF(s).fit(X_,y_)
    if name=='XGBoost':
        return XGBClassifier(n_estimators=150,max_depth=6,learning_rate=0.05,scale_pos_weight=ratio,subsample=0.8,
                             colsample_bytree=0.8,eval_metric='logloss',random_state=s,verbosity=0).fit(X_train,y_train)
NAMES = ['Baseline RF','SMOTE+RF','BalancedRF','SMOTETomek+RF','XGBoost']
seed_res = {n:[] for n in NAMES}
for n in NAMES:
    for s in SEEDS:
        m = build(n,s); seed_res[n].append(ev(n,y_test,m.predict(X_test),m.predict_proba(X_test)[:,1]))
    print('done', n)
rows=[]
for n in NAMES:
    d={'Model':n}
    for k in ['P','R','F1','AUC','FDR']:
        v=[r[k] for r in seed_res[n]]; d[k]=f'{np.mean(v):.3f} +- {np.std(v):.3f}'
    rows.append(d)
print(pd.DataFrame(rows).to_string(index=False))

print('\nPaired tests vs Baseline RF (n=5 seeds)')
for k in ['F1','AUC','FDR']:
    base=[r[k] for r in seed_res['Baseline RF']]
    for n in NAMES[1:]:
        o=[r[k] for r in seed_res[n]]
        t=ttest_rel(base,o)
        try: w=wilcoxon(base,o).pvalue
        except ValueError: w=float('nan')
        print(f'{k:3s} Baseline vs {n:14s} t-test p={t.pvalue:.4f}  Wilcoxon p={w:.4f}')
print('\nNOTE: with n=5 the smallest possible two-sided Wilcoxon p is 0.0625, so it cannot reach 0.05. Report the paired t-test.')


In [ ]:
# 12c. Siena threshold sensitivity (Reviewers 2 and 3: low cross-dataset recall)
# NOTE: sweeping the threshold on Siena labels is an ORACLE analysis (an upper bound on what recalibration
# could give), not a deployable result. Say so in the paper.
ths=[0.50,0.40,0.30,0.20,0.10,0.05,0.02]
out=[]
for name,model in [('Baseline RF',m1),('SMOTE+RF',m2),('BalancedRF',m3),('SMOTETomek+RF',m4),('XGBoost',m5)]:
    pp=model.predict_proba(X_sp)[:,1]
    for t in ths:
        r=ev(name,y_s,(pp>=t).astype(int),pp); r['thr']=t; out.append(r)
df_th=pd.DataFrame(out)[['Model','thr','P','R','F1','FDR']]
print(df_th.to_string(index=False))
print('\nBest-F1 threshold per model:')
print(df_th.loc[df_th.groupby('Model').F1.idxmax()].to_string(index=False))
